# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR² dataset (clinicopathological and molecular characteristics of second primary colorectal cancer) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to entities such as record sets, fields, and columns use their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is described via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and examine the high-level description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Here we enumerate all available [Record Sets](https://github.com/mlcommons/croissant/blob/main/docs/specification.md#recordset), and their fields/columns, referencing all by their `@id`.

**Note:** If record sets are not listed at the package level, you can use `dataset.record_sets` to list them.

In [ ]:
# List all record sets (by @id) in the dataset
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in dataset.")
else:
    print(f"{len(record_sets)} record set(s) detected:\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[No name]')}")
        print(f"  Description: {rs.get('description', '[No description]')}")
        # List fields/columns (if present)
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if len(fields) == 0:
            print("    No fields in this record set.")
        else:
            print(f"    Fields/Columns:")
            for field in fields:
                # Each field might be a link (@id) or a dict
                if isinstance(field, str):
                    print(f"      - {field}")
                elif isinstance(field, dict):
                    print(f"      - {field.get('@id', '[no id]')}: {field.get('name', '[no name]')}")
        print("")

## 3. Data Extraction
Load all data from one or more record sets (referenced strictly by their `@id`) into pandas DataFrames for analysis. Below, we assemble a list of all record set `@id`s; you may refine the list if the dataset is large.

In [ ]:
# Create a mapping of record_set @id -> DataFrame
df_map = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"  Number of records: {len(df)} | Columns: {list(df.columns)}\n")
        df_map[rs_id] = df
    except Exception as e:
        print(f"  Failed to load: {e}\n")

# As illustration, show columns and a sample of the first DataFrame (if any records exist)
if len(df_map):
    first_rs = record_set_ids[0]
    print(f"Columns in first RecordSet (@id: {first_rs}):\n{df_map[first_rs].columns.tolist()}")
    display(df_map[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (column) by its `@id` from the loaded DataFrame for summary, filtering, normalization, and grouping operations. 

If column `@id`s are not available, use column names as mapped from record set/field inspection above.

In [ ]:
# For demonstration, choose the primary data table RecordSet (by @id)
main_rs_id = record_set_ids[0] if len(record_set_ids) else None
if main_rs_id is None or main_rs_id not in df_map:
    print("No record set data to analyze.")
else:
    df = df_map[main_rs_id]

    # 1. Identify a numeric field (e.g., patient age or interval in months if present)
    numeric_field = None
    for c in df.columns:
        col_lower = str(c).lower()
        if ('age' in col_lower or 'interval' in col_lower or 'months' in col_lower or 'year' in col_lower or 'num' in col_lower or 'count' in col_lower):
            # Check if column is numeric
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field = c
                break
    if numeric_field is None:
        # Try all columns for numeric
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if num_cols:
            numeric_field = num_cols[0]

    if numeric_field:
        print(f"Selected numeric field: {numeric_field}")
        # 2. Apply threshold filtering
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records (where {numeric_field} > {threshold:.2f}): {len(filtered_df)} found.")

        # 3. Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst 5 normalized values:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # 4. Group by a categorical field if available
        group_field = None
        for c in df.columns:
            if (c != numeric_field and df[c].dtype == object and pd.Series(df[c]).nunique() < len(df) // 2):
                group_field = c
                break
        if group_field:
            print(f"\nGrouping by field: {group_field}\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean')
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping.")
    else:
        print("No numeric field found in DataFrame.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and (if grouping was possible) compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Group mean plot
    if 'group_field' in locals() and group_field:
        group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=group_means)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

This notebook demonstrates the workflow for loading, exploring, and analyzing a dataset structured by the Croissant schema using the `mlcroissant` Python library. Through referencing all data entities via their `@id`, you gain complete traceability and reproducibility when inspecting fields, record sets, and analytic steps.

_For more advanced analyses, consider exploring relationships between fields, longitudinal trends, or exporting the processed DataFrames for machine learning or reporting._